In [0]:
import sys
import os

current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, "..","..",".."))

sys.path.append(project_root)

In [0]:
from pyspark.sql.functions import *
from src.citybike.citybike_utils import get_trip_duration_mins
from src.utils.datetime_utils import timestamp_to_date_col

In [0]:
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")
catalog = dbutils.widgets.get("catalog")

In [0]:
df = spark.read.table(f"{catalog}.01_bronze.jc_citybike")

In [0]:
df = get_trip_duration_mins(spark, df , "started_at", "ended_at", "trip_duration_mins")

In [0]:
df = timestamp_to_date_col(spark, df, "started_at", "trip_start_date")

In [0]:
df = df.withColumn("metadata",
                   create_map(
                       lit("pipeline_id"), lit(pipeline_id),
                       lit("run_id"), lit(run_id),
                       lit("task_id"), lit(task_id),
                       lit("processed_timestamp"), lit(processed_timestamp)
                   ))

In [0]:
df = df.select("ride_id", "trip_start_date", "started_at", "ended_at", "start_station_name", "end_station_name","trip_duration_mins","metadata")

In [0]:
df.write\
    .mode("overwrite")\
    .option("overwriteSchema",True)\
    .saveAsTable(f"{catalog}.02_silver.jc_citybike")